# Metadata Filtering in LlamaIndex

Metadata filtering lets you narrow retrieval to a specific subset of documents — e.g. only `category: ufotable` docs — _before_ semantic search runs, instead of hoping the embedding similarity alone finds the right section.


Standard setup — quiet logging, load env vars, and set the default LLM/embedding model.


In [1]:
import logging

from dotenv import load_dotenv
from llama_index.core import Settings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI

# Quiet noisy INFO-level logs from LlamaIndex and its HTTP client.
for noisy_logger in ("httpx", "llama_index"):
    logging.getLogger(noisy_logger).setLevel(logging.WARNING)

# Load API keys from .env into the environment.
load_dotenv()

# Set the default LLM and embedding model used everywhere in this notebook.
Settings.llm = OpenAI(model="gpt-4.1-nano")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

**Step 1 — Attach custom metadata.** Tag each document with the animation studio that produced it. This metadata carries through to every node created from that document, which is what makes filtering possible in the next step.


In [2]:
from llama_index.core import SimpleDirectoryReader

# Map each source file to the animation studio that produced it — this becomes
# metadata we can filter on later.
category_by_filename = {
    "naruto.txt": "Pierrot",
    "dragon_ball.txt": "Toei Animation",
    "solo_leveling.txt": "A-1 Pictures",
    "death_note.txt": "Madhouse",
    "demon_slayer.txt": "ufotable",
}

documents = SimpleDirectoryReader("data/sample_docs").load_data()
for document in documents:
    filename = document.metadata["file_name"]  # SimpleDirectoryReader adds this automatically
    document.metadata["category"] = category_by_filename[filename]  # attach our custom metadata
    print(f"{filename} -> category={document.metadata['category']}")

death_note.txt -> category=Madhouse
demon_slayer.txt -> category=ufotable
dragon_ball.txt -> category=Toei Animation
naruto.txt -> category=Pierrot
solo_leveling.txt -> category=A-1 Pictures


**Step 2 — Filter by metadata before searching.** Build the index, then compare an unfiltered query against the same query restricted to `category="ufotable"` — notice how the filtered version only pulls from the Demon Slayer document.


In [3]:
from llama_index.core import VectorStoreIndex
from llama_index.core.vector_stores import ExactMatchFilter, MetadataFilters

# Metadata now lives on every node too (inherited from its parent Document),
# so the index can filter on it before doing any similarity search.
index = VectorStoreIndex.from_documents(documents)
question = "What technique or ability does the protagonist use to become stronger?"

# Baseline: search across ALL documents regardless of category.
unfiltered_response = index.as_query_engine(similarity_top_k=3).query(question)

# ExactMatchFilter restricts retrieval to nodes whose "category" metadata equals
# "ufotable" (i.e. only the Demon Slayer document) — applied BEFORE similarity search runs.
studio_filters = MetadataFilters(filters=[ExactMatchFilter(key="category", value="ufotable")])
filtered_response = index.as_query_engine(similarity_top_k=3, filters=studio_filters).query(question)

print("--- WITHOUT metadata filter ---")
print(unfiltered_response)
print(f"Categories retrieved: {[n.metadata['category'] for n in unfiltered_response.source_nodes]}\n")

print("--- WITH category=ufotable filter ---")
print(filtered_response)
print(f"Categories retrieved: {[n.metadata['category'] for n in filtered_response.source_nodes]}")

--- WITHOUT metadata filter ---
The protagonist uses a unique system that allows him to level up, complete quests, and grow stronger in ways no other hunter can. He also has the ability to summon loyal shadow soldiers, including named shadows, which enhances his strength.
Categories retrieved: ['ufotable', 'A-1 Pictures', 'Toei Animation']

--- WITH category=ufotable filter ---
The protagonist, Tanjiro Kamado, initially fights using Water Breathing, a swordsmanship style that mimics the fluid, adaptable properties of water. Later, he awakens Hinokami Kagura, also known as Sun Breathing, which is the original breathing style from which all other styles descend.
Categories retrieved: ['ufotable', 'ufotable']


### Summary

- Metadata filters narrow the search space _before_ semantic matching runs, which both speeds up retrieval and avoids relevant-looking-but-wrong-category results leaking into an answer.
